In [25]:
import openai
from pinecone import Pinecone, ServerlessSpec          # v3 client
from langchain_pinecone import PineconeVectorStore     # new LC wrapper
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.docstore.document import Document
from dotenv import load_dotenv
import os

load_dotenv()
# set API key
OPENAI_KEY=os.getenv("OPENAI_API_KEY")
OPENAI_BASE=os.getenv("OPENAI_API_BASE")
PINECONE_KEY = os.getenv('PINECONE_API_KEY')


openai_client = openai.OpenAI(
    base_url = OPENAI_BASE,
    api_key=OPENAI_KEY)


In [41]:
# get the history data 
from google.cloud import bigquery
client=bigquery.Client(project='khanacademy.org:deductive-jet-827')
query = """
with lib as (
SELECT DISTINCT
      content_path.content_slug,
      content_path.content_id
    FROM `khan-core.content.published_content_paths_daily`
    WHERE dt = '2025-10-06'
      AND locale = 'en'
      and content_path.domain_slug='math'
      
      )
, threads as ( 
select distinct a.userKaid as kaid,
 a.aiGuideThreadId, 
 a.contentId,
 a.contentKind, 
 a.delta_dt,
 b.content_slug
 FROM `khan-data-lake.analytics_events.AiGuideInteractionStored_materialized` a
left join lib b
 
on a.contentId=b.content_id 
where a.eventInfo.eventTime>'2025-08-05'
and a.aiGuideExperienceReportingLabelId='content-tutoring'
and a.userKaid='kaid_248328473686127753888179'
)
, text as (
select kaid, 
threadId, 
string_agg(concat(conversationEntryType, ": ", text), " ") as thread_text
from (
    select kaid,
    threadId, 
    conversationEntryType, text
    from `khan-iris.aiguide.conversation_entries`
    where dt > '2025-08-05'
    and threadID in (select aiGuideThreadId from threads)
    --AND threadID in (select aiGuideThreadId from user_lengths)
    order by threadID, conversationEntryTime
) group by 1, 2  
)
-- need tp adjust fpm to just get current state at today
, fpm AS (
  SELECT distinct
    a.kaid,
    a.skill_id,
  -- a.skill_fpm_before_level,
  --  a.skill_fpm_after_level
    LAST_VALUE(a.skill_fpm_after_level) OVER (PARTITION BY a.kaid, skill_id ORDER BY a.fpm_event_ts ASC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_fpm_level
    FROM `khan-core.mastery.fpm_events` as a
    where fpm_event_dt between '2025-08-05' and CURRENT_DATE()
    and kaid in (select kaid from threads)
  and task_type= 'practice_tutorial'
)
select
  kaid, 
  aiGuideThreadId as turn_id,
  content_slug as skill_tag,
  delta_dt as dt,
  last_fpm_level as fpm_state,
  thread_text as text
  from (
select threads.*, 
  text.thread_text,
  --fpm.skill_fpm_before_level,
  fpm.last_fpm_level  
  from threads 
  left join text on 
  threads.kaid=text.kaid and 
  threads.AiGuideThreadId = text.threadId
  left join fpm on 
  threads.kaid=fpm.kaid and 
  threads.contentID = fpm.skill_id
  ) 
 """

# query with prereqs and fpm on those
query2="""
with lib as (
SELECT DISTINCT
      content_path.content_slug,
      content_path.content_id
    FROM `khan-core.content.published_content_paths_daily`
    WHERE dt = '2025-10-06'
      AND locale = 'en'
      and content_path.domain_slug='math'
      
      )
, threads as ( 
select distinct a.userKaid as kaid,
 a.aiGuideThreadId, 
 a.contentId,
 a.contentKind, 
 a.delta_dt,
 b.content_slug
 FROM `khan-data-lake.analytics_events.AiGuideInteractionStored_materialized` a
left join lib b
 
on a.contentId=b.content_id 
where a.eventInfo.eventTime>'2025-08-05'
and a.aiGuideExperienceReportingLabelId='content-tutoring'
and a.userKaid='kaid_248328473686127753888179'
)
, text as (
select kaid, 
threadId, 
string_agg(concat(conversationEntryType, ": ", text), " ") as thread_text
from (
    select kaid,
    threadId, 
    conversationEntryType, text
    from `khan-iris.aiguide.conversation_entries`
    where dt > '2025-08-05'
    and threadID in (select aiGuideThreadId from threads)
    --AND threadID in (select aiGuideThreadId from user_lengths)
    order by threadID, conversationEntryTime
) group by 1, 2  
)
-- need tp adjust fpm to just get current state at today
, fpm AS (
  SELECT distinct
    a.kaid,
    a.skill_id,
  -- a.skill_fpm_before_level,
  --  a.skill_fpm_after_level
    LAST_VALUE(a.skill_fpm_after_level) OVER (PARTITION BY a.kaid, skill_id ORDER BY a.fpm_event_ts ASC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_fpm_level
    FROM `khan-core.mastery.fpm_events` as a
    where fpm_event_dt between '2025-08-05' and CURRENT_DATE()
    --and kaid='kaid_248328473686127753888179'
    and kaid in (select kaid from threads)
  and task_type= 'practice_tutorial'
) 
, prereqs as (select a.*,
 lib.content_id as prereq_content_id from (
 SELECT
  learnableContentRevision.slug,
  learnableContentRevision.contentId,
  json_value(prereq_element) AS prerequisite,
  MAX(eventInfo.eventTime) AS latest_version,
FROM
  `khan-data-lake.analytics_events.ExerciseRevisionPublish_materialized`  ,
  UNNEST(JSON_EXTRACT_ARRAY(prerequisitesJson)) AS prereq_element
WHERE
  learnableContentRevision.kaLocale = 'en'
GROUP BY
  learnableContentRevision.slug,
  learnableContentRevision.contentId,
  prereq_element
) a
  left join lib on 
  lib.content_slug = a.prerequisite
  ) 
, prereq_fpm as (
  SELECT distinct
    a.kaid,
    a.skill_id as prereq_content_id,
  -- a.skill_fpm_before_level,
  --  a.skill_fpm_after_level
    LAST_VALUE(a.skill_fpm_after_level) OVER (PARTITION BY a.kaid, skill_id ORDER BY a.fpm_event_ts ASC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS prereq_fpm_level
    FROM `khan-core.mastery.fpm_events` as a
    where fpm_event_dt between '2024-08-05' and CURRENT_DATE()
    --and kaid='kaid_248328473686127753888179'
    and kaid in (select kaid from threads)
    and skill_id in (select prereq_content_id from prereqs)
    and task_type= 'practice_tutorial'
)
, hist as (
select
  kaid, 
  aiGuideThreadId as turn_id,
  contentId,
  content_slug as skill_tag,
  delta_dt as dt,
  last_fpm_level as fpm_state,
  thread_text as text
  from (
select threads.*, 
  text.thread_text,
  --fpm.skill_fpm_before_level,
  fpm.last_fpm_level
  from threads 
  left join text on 
  threads.kaid=text.kaid and 
  threads.AiGuideThreadId = text.threadId
  left join fpm on 
  threads.kaid=fpm.kaid and 
  threads.contentID = fpm.skill_id
  )
)
select 
  kaid, 
  turn_id,
  skill_tag,
  dt, 
  fpm_state,
  prerequisite,
  case when prereq_fpm_level is null then 'unknown' else prereq_fpm_level end as prereq_fpm_level,
  text
  from (
select hist.*,
b.prereq_content_id,
b.prerequisite ,
c.prereq_fpm_level
from hist 
left join prereqs b 
on hist.contentId = b.contentId
left join prereq_fpm c 
on b.prereq_content_id = c.prereq_content_id
)
"""

In [ ]:
# create vector DB and upsert docs
INDEX_NAME       = "student"
DIMENSION        = 1536                    
METRIC           = "cosine"                

# init pinecone 
pc = Pinecone(api_key=PINECONE_KEY)
# create index  
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(cloud="gcp", region="us-central1")
    )

index = pc.Index(INDEX_NAME)

# # Initialize the embedder
embedder = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_KEY
)
# create the vector store 
vector_store = PineconeVectorStore(index, embedder)

# build and upload docs 
def build_docs(rows):
    docs=[]
    for kaid, turn_id, skill_tag, dt, fpm_state, prerequisite, prereq_fpm_level, text in rows:
        metadata={'kaid': kaid, 
                  'turn_id': turn_id, 
                  'skill_tag': skill_tag, 
                  'date': str(dt), 
                  'fpm_state': fpm_state,
                  'prerequisite': prerequisite,
                  'prereq_fpm_level': prereq_fpm_level}
        docs.append(Document(page_content=text, metadata=metadata))
    return docs

rows=client.query(query2)
docs = build_docs(rows)

vector_store.add_documents(docs)

        

In [ ]:
# create the memory block  (try "multiplication_3", or "regrouping-whole-numbers" )
# multiply-with-partial-products--2-digit-numbers-  # -- good example of familiar prereq
def build_memory_block(student_msg, kaid, skill_tag, k=4 ):

  vec_results = vector_store.similarity_search(student_msg, 
                                               k=k, 
                                               filter=({"kaid" : kaid, "skill_tag": skill_tag})
                                               )
  memory_chunks = []
  skill_state = {}
  prereq_state ={} 
  
  for doc in vec_results:
      m = doc.metadata
      # get past text 
      turns   = m.get("turn_id")
      skills  = m.get("skill_tag", [])
      dt = m.get("date", "")

      header = f"[turn {turns} | skills: {skills} | {dt}]"
      chunk  = f"{header}\n{doc.page_content}"
      memory_chunks.append(chunk)

      # get skill levels for current skill 
      skill  = m.get("skill_tag", [])
      skill_level = m.get("fpm_state", "")
      skill_state[skill] = skill_level
  
      # get prereq levels 
      prereq = m.get("prerequisite", "")
      prereq_level = m.get("prereq_fpm_level", "unknown")
      prereq_state[prereq] = prereq_level

  memory_block = "\n\n".join(memory_chunks)          
  
  skill_lines = [f"{skill}: {level}" for skill, level in skill_state.items()]
  skill_block = "\n".join(skill_lines)  
  
  prereq_lines = [f"{prereq}: {level}" for prereq, level in prereq_state.items()]
  prereq_block = "\n".join(prereq_lines)     
     
  return memory_block, skill_block, prereq_block
  

In [ ]:
student_msg = 'I am having trouble using partial products to multiply 85 times 61'

system_prompt = ("""
# ROLE 
"You are an AI math tutor.
# YOUR PEDAGOGY RULES
1. Use Socratic questioning.
2. Encourage student to explain their reasoning aloud.
3. Do not do the work for the student and do not give away answers. 
# ADDITIONAL CONTEXT
You will be provided with three items: 
    1 - the history of past conversations between tutor and the student on this topic 
    2 - student skill level on the given skill that you are tutoring
    3 - student skill level on the prerequisite skills that are needed to master the skill you are tutoring.                  
              
Use this information to adjust your tutoring approach as follows:
1. First look at the prerequisite proficiency. If any prerequisite skill is at "familiar" level, tell the student that they should review \
                 that skill first and offer a link to the video.  
2. Then state students proficiency level on the current skill.
3. If you see in the conversation history that the student already received help on this question, \
remind them that you already discussed this and ask if they remember that.  
4. If student is proficient on a skill acknowledge that and offer minimal support, like a broad strategy hint.
5. If a student is familiar or attempted on a skill, provide detailed support step by step

Here is an example:
    Student: How do I find common denominators?
    Tutor: It looks like you may need to first review an important prerequisite \
    Here is a link to a video you should watch first, and then we can do a quick practice problem on that.
    I see that you are familiar with finding common denominators. ALso I see you asked about this before. Do you remember our previous discussion?                 
""")


# test memory input for tutor 
def tutor(student_msg, kaid, skill_tag,  model="gpt-4-khan", k=4, temperature=0.0):
    # get memory block and skill block for this case
    memory_block, skill_set, prereq_set = build_memory_block(student_msg, kaid, skill_tag, k=k)
    
        
    completion = openai_client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "system", "content": 'the current knowldfge state on the relevant skill: ' + skill_set },
            {"role": "system", "content": 'relevant past conversations :'  + memory_block },
            {"role": "system", "content": 'relevant past conversations :'  + prereq_set },
                        
            {"role": "user", "content": student_msg}
            ]) 
    return completion.choices[0].message.content
    

tutor(student_msg, 'kaid_248328473686127753888179', ' multiply-with-partial-products--2-digit-numbers-', k=4)

"I see that you are familiar with the concept of multiplication, which is a prerequisite for understanding partial products. However, you may need to review the concept of place value, which is also crucial for understanding partial products. Here is a link to a video you should watch first: [Place Value Video](https://www.youtube.com/watch?v=5W47G-h7myY)\n\nAfter reviewing place value, let's revisit your question. I see that you've asked about using partial products to multiply before. Do you remember our previous discussion on this topic? \n\nIf you're still having trouble, let's break it down. When you're multiplying 85 by 61 using partial products, what's the first step you should take?"

In [ ]:
# convo simulator to test next and need to add a classifier to student question to detect skill

# def tutor(question, student_response_prompt=student_response_prompt, model="gpt-4-khan", system=None):
#     counter = 0
#     tutor_responses=[]
#     student_questions =[question]
#     while counter<3:
#         if counter==0:
#             completion = openai_client.chat.completions.create(
#                 model=model,
#                 temperature=0.0,
#                 messages=[
#                     {"role": "system", "content": system_prompt},
#                     {"role": "user", "content": question}
#                     ]) # ask first question
#             # store tutor response
#             tutor_responses.append(completion.choices[0].message.content)
            
#         else:
#             # generate next student question
#             student_q = student(tutor_responses[-1], student_response_prompt)
#             student_questions.append(student_q)
#             completion = openai_client.chat.completions.create(
#                 model=model,
#                 temperature=0.0,
#                 messages=[{"role": "user", "content": student_q}])
#             # generate tutor response to next question
#             tutor_responses.append(completion.choices[0].message.content)
                       
#         counter += 1
#     return pd.DataFrame({'question': student_questions, 'responses': tutor_responses})

# def student(response, student_response_prompt,  model="gpt-4-khan", system=None):
#     responses=[]
#     completion = openai_client.chat.completions.create(
#         model=model,
#         temperature=0.0,
#         messages=[{"role": "user", "content": student_response_prompt + response}])
#     response = completion.choices[0].message.content
#     responses.append(response)
#     return responses[-1]

# tutor(question)
